

$$ max_{virtuais} = (n-1)\cdot d \cdot Q^{,d-1} $$


onde

* (n) = número de amostras reais,
* (d) = número de entradas (dimensões),
* (Q) = número de quantis (Q_a) usados (isto é, o número de valores de (a) que você fixa para as outras dimensões). 

Aplicando aos seus valores:

* (n=25) (25 amostras) → (n-1=24)
* (d=10) (10 entradas)
* se usar o conjunto sugerido ({0.025,0.25,0.5,0.75,0.975}) → (Q=5)


$$ max_{virtuais} = 24 \times 10 \times 5^{9} = 468,750,000 $$


Ou seja, **468 750 000** amostras virtuais no máximo (explosão combinatória enorme).

Observações práticas importantes

* Esse número é teórico — na prática é inviável computacionalmente e muitas dessas combinações podem ser irrelevantes ou redundantes.
* Os autores mencionam justamente essa explosão e, em aplicações reais, costumam reduzir (Q) (por exemplo, usar somente (a=0.5), i.e. (Q=1)) para diminuir o custo. Com (Q=1) o mesmo cálculo dá (24\times 10 \times 1^{9}=240) (ordem de centenas), muito mais manejável — e coincide com a motivação do artigo para escolher menos quantis em problemas reais. 
* Outro ponto: o número de saídas (1 saída no seu caso) **não** entra na fórmula — só importam entradas/dimensões, quantis e (n).


In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ExpSineSquared, DotProduct, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
out_scaler = StandardScaler()

In [2]:

PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "IP"] # 10 entradas

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

In [3]:
Data = pd.read_excel("./Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Data[Data["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])

    Datasets.append(n_data)

    

# Métricas

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import os

def PlotPredictions(train_orig, train_pred, test_orig, test_pred, target_name, n): 
    # cria figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -----------------------------
    # SUBPLOT 1 — TREINO
    # -----------------------------
    ax = axes[0]
    n_train = len(train_orig)
    x_train = np.arange(n_train)

    ax.plot(x_train, train_orig, label="Original (train)", color="blue", )
    ax.plot(x_train, train_pred, label="Predito (train)", color="red",)

    ax.scatter(x_train, train_orig, color="blue", s=35)
    ax.scatter(x_train, train_pred, color="red", s=35)

    ax.set_title("Treinamento")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    # -----------------------------
    # SUBPLOT 2 — TESTE
    # -----------------------------
    ax = axes[1]
    n_test = len(test_orig)
    x_test = np.arange(n_test)

    ax.plot(x_test, test_orig, label="Original (test)", color="blue", linewidth=1.8)
    ax.plot(x_test, test_pred, label="Predito (test)", color="red", linewidth=1.8)

    ax.scatter(x_test, test_orig, color="blue", s=35)
    ax.scatter(x_test, test_pred, color="red", s=35)

    ax.set_title("Teste")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    plt.tight_layout()

    # salva a figura
    filename = f"./Dados/VirtualData/P{n}/TrainResults/{target_name}.pdf"
    plt.savefig(filename, format="pdf", bbox_inches="tight")

    # fecha a figura (IMPORTANTE para não acumular memória)
    plt.close(fig)

    print(f"Figura salva em: {filename}")


In [5]:
from sklearn.metrics import r2_score, mean_squared_error 

def ComputeMetrics(model, df_train, df_test, target, n):
    
    test_pred = out_scaler.inverse_transform(model.execute(df_test[PREDICTORS]).reshape(-1, 1))
    train_pred = out_scaler.inverse_transform(model.execute(df_train[PREDICTORS]).reshape(-1, 1))
    
    test_orig = out_scaler.inverse_transform(df_test[[target]])
    train_orig = out_scaler.inverse_transform(df_train[[target]])
    
    PlotPredictions(train_orig, train_pred, test_orig, test_pred, target, n)
    return {
        'r2_test': r2_score(test_orig, test_pred),
        'mse_test': mean_squared_error(test_orig, test_pred),
        'r2_train': r2_score(train_orig, train_pred),
        'mse_train': mean_squared_error(train_orig, train_pred),
    }

# Extraindo dados

In [6]:
Krbf = C(1.0) * RBF(length_scale=1.0, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Kmtrn = C(1.0) * Matern(length_scale=1.0, nu=0.5, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Krq = C(1.0) * RationalQuadratic(length_scale=1.0, alpha=0.1, length_scale_bounds=(1e-9, 1e+6)) # 
Kess = C(1.0) * ExpSineSquared(length_scale=1.0, periodicity=3.0, length_scale_bounds=(1e-9, 1e+6)) 
Kdp = C(1.0) * DotProduct() + WhiteKernel()

In [7]:

def PrepareData(Dataset, target):
    scaler.fit(Dataset[PREDICTORS])
    out_scaler.fit(Dataset[[target]])
    
    TrainData, TestData = train_test_split(Dataset, test_size=0.2, random_state=42)
    NormTrainData, NormTestData = TrainData, TestData 
    
    NormTrainData[PREDICTORS] = scaler.transform(TrainData[PREDICTORS])
    NormTrainData[target] = out_scaler.transform(TrainData[[target]])
    
    NormTestData[PREDICTORS] = scaler.transform(TestData[PREDICTORS])
    NormTestData[target] = out_scaler.transform(TestData[[target]])
    
    return TrainData, NormTrainData, TestData, NormTestData


In [8]:
kernel =  Kmtrn

class GPRVSG:
    def __init__(self, Dataset, target):
        self.target = target
        self.TrainData, self.NormTrainData, self.TestData, self.NormTestData = PrepareData(Dataset,
                                                                                           target)
        self.x_train =  self.NormTrainData[PREDICTORS].values
        self.y_train = self.NormTrainData[target].values
        
        self.x_test = self.NormTestData[PREDICTORS].values
        self.y_test = self.NormTestData[target].values
        
        self.d = self.x_train.shape[1]
        self.virtual_samples_x = []
        self.virtual_samples_y = []

    def ComputeProjection(self):
        projections = []
        for m in range(self.d):
            x_m = self.x_train[:, m]         # valores reais da dimensão m
            s_m = np.sort(x_m)               # projeções ordenadas
            projections.append(s_m)
        return projections


    def SetInputSpace(self):
        projections = self.ComputeProjection()
        avg_dists = [np.mean(np.diff(proj)) for proj in projections]

        Q_alpha = np.quantile(projections, [0.5], axis=1, method="hazen").T
        for m in range(self.d):
            for i in range(len(projections[m]) - 1):
                dist = projections[m][i + 1] - projections[m][i]
                if dist > avg_dists[m]:
                    G = (projections[m][i] + projections[m][i + 1]) / 2
                    for q in range(self.d):
                        if q != m:
                            for quantile in Q_alpha[q]:
                                tilde_q = np.zeros(self.d)
                                tilde_q[m] = G
                                tilde_q[q] = quantile
                                self.virtual_samples_x.append(tilde_q)
        self.virtual_samples_x = np.array(self.virtual_samples_x)

    def BuildModel(self):
        self.model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=100,
                                              alpha=1e-8, normalize_y=True)
        self.model.fit(self.x_train, self.y_train)

    def getX(self):
        n_dims = self.virtual_samples_x.shape[1]
        col_names = [f"x{i+1}" for i in range(n_dims)]
        df = pd.DataFrame(self.virtual_samples_x, columns=col_names)
        return [df[col] for col in col_names]

    def ComputeY(self,):
        # pega lista de colunas: [x1, x2, ..., xn]
        X_cols = self.getX()

        # monta matriz X de entrada
        X = np.column_stack(X_cols)

        # prediz y
        self.virtual_samples_y = self.model.predict(X)

        # cria nomes x1, x2, ..., xn
        n_dims = len(X_cols)

        # monta DataFrame final
        data = {name: X_cols[i] for i, name in enumerate(PREDICTORS)}
        data[self.target] = self.virtual_samples_y

        self.virtual_samples_df = pd.DataFrame(data)

    def execute(self, *coords):
        # transformar lista de vetores em matriz X
        X = np.column_stack(coords)

        y_pred = self.model.predict(X)
        return y_pred

    def Run(self):
        self.SetInputSpace()
        self.BuildModel()
        self.ComputeY()

In [9]:
def PlotVirtualData(virtual_df, original_df, predictors, target, n):
    savepath = f"./Dados/VirtualData/P{n+1}/VSGResults/Virtual_{target}.pdf"

    # total = 10 entradas + 1 saída
    total_features = len(predictors) + 1
    fig, axes = plt.subplots(total_features, 1, figsize=(10, 2*total_features), sharex=False)

    # junta entradas + saída
    features = predictors + [target]

    for i, feat in enumerate(features):

        ax = axes[i]

        # valores preditos (virtuais)
        y_pred = virtual_df[feat].values
        x_pred = np.arange(len(y_pred))

        # valores originais reais (Dataset)
        y_orig = original_df[feat].values
        x_orig = np.arange(len(y_orig))

        # plot
        ax.plot(x_pred, y_pred, color="red", label="Virtual (predito)", linewidth=1.7)
        ax.scatter(x_pred, y_pred, color="red", s=20)

        ax.plot(x_orig, y_orig, color="blue", label="Original", linewidth=1.7)
        ax.scatter(x_orig, y_orig, color="blue", s=20)

        ax.set_ylabel(feat)
        ax.grid(True)

        if i == 0:
            ax.legend()

    axes[-1].set_xlabel("Amostras")

    plt.tight_layout()
    plt.savefig(savepath, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Figura salva em: {savepath}")


In [10]:

import os

def GetVirtualData():
    for n in range(0, 4):  
        metrics_all = []
        output_dir = f"./Dados/VirtualData/P{n+1}"
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/TrainResults/", exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/VSGResults/", exist_ok=True)

        for target in TARGETS:            
            gpr_vsg = GPRVSG(Datasets[n], target)
            gpr_vsg.Run()
            metrics = ComputeMetrics(gpr_vsg,gpr_vsg.NormTrainData,gpr_vsg.NormTestData,target,n+1)

            vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
            gpr_vsg.virtual_samples_df.to_excel(vs_filename, index=False)

            # Guarda métricas + nome do target
            metrics_all.append(metrics)
            metrics["target"] = target  
            
            PlotVirtualData(
                virtual_df = gpr_vsg.virtual_samples_df,          # dados virtuais desnormalizados
                original_df = Datasets[n],                        # dataset original desnormalizado
                predictors = PREDICTORS,                          # entradas
                target = target,                                  # saída
                n = n
            )
            # break
        # Converte lista de dicts para DataFrame
        df_metrics = pd.DataFrame(metrics_all)
        display(df_metrics)

        # Salva arquivo final por ponto
        metrics_filename = os.path.join(output_dir, "GprMetrics.xlsx")
        df_metrics.to_excel(metrics_filename, index=False)

In [11]:
GetVirtualData()

Figura salva em: ./Dados/VirtualData/P1/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_

,r2_test,mse_test,r2_train,mse_train,target
0,0.782044,31061.758866,1.0,3.626025e-11,Fe
1,-14.008424,6186.850650,1.0,1.109950e-12,Al
2,-0.222180,0.000606,1.0,1.663002e-19,As
3,-19.228869,0.198764,1.0,4.448001e-17,Pb
4,-0.000509,513.066393,1.0,7.688005e-14,Zn
5,-13.060636,0.003019,1.0,1.713087e-18,Hg
6,0.301182,0.046393,1.0,8.596022e-18,Co
7,0.560133,0.019668,1.0,1.308391e-17,V
8,0.295583,39.747946,1.0,1.314023e-14,Ba
9,-0.029335,3428.675064,1.0,3.970404e-13,Mn


Figura salva em: ./Dados/VirtualData/P2/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Zn.pdf


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-09. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P2/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_V.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Ba.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Ba.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Mn.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Mn.pdf


...

,r2_test,mse_test,r2_train,mse_train,target
0,0.355200,224583.150622,1.0,3.083535e-11,Fe
1,-21.595288,13046.695507,1.0,2.587224e-12,Al
2,0.537267,0.001099,1.0,3.418218e-19,As
3,-0.578145,0.060635,1.0,2.284149e-17,Pb
4,-0.013131,1009.011820,1.0,3.359030e-14,Zn
5,-29.983335,0.008889,1.0,1.466777e-17,Hg
6,0.096530,0.051135,1.0,6.412390e-18,Co
7,0.347867,0.033898,1.0,1.281026e-17,V
8,-0.067697,55.701365,1.0,1.688424e-14,Ba
9,0.355526,1874.813591,1.0,2.291374e-13,Mn


Figura salva em: ./Dados/VirtualData/P4/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Zn.pdf


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-09. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Figura salva em: ./Dados/VirtualData/P4/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_V.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Ba.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Ba.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Mn.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Mn.pdf


,r2_test,mse_test,r2_train,mse_train,target
0,0.223281,225686.083551,1.0,3.523550e-11,Fe
1,-1.449209,4808.754033,1.0,4.669722e-12,Al
2,0.445214,0.001062,1.0,1.268071e-18,As
3,-9.632027,0.122050,1.0,9.787304e-17,Pb
4,-0.090560,496.069795,1.0,3.923873e-14,Zn
5,-0.016334,0.007120,1.0,5.526643e-19,Hg
6,0.147894,0.017828,1.0,3.861592e-18,Co
7,0.505161,0.018803,1.0,1.584818e-17,V
8,-0.237033,65.859941,1.0,5.726918e-14,Ba
9,0.313487,906.637549,1.0,1.775809e-13,Mn
